In [18]:
%run ../src/dinobot/clasificador_intenciones.py



In [16]:

import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


BASE_DIR = Path.cwd().resolve()

while not (BASE_DIR / "src" / "dinobot" / "clasificador_intenciones.py").is_file():
    if BASE_DIR == BASE_DIR.parent:
        raise FileNotFoundError(
            "Abre el notebook desde la carpeta DinoBot o una de sus subcarpetas."
        )

    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR / "src" / "dinobot"))
from clasificador_intenciones import intenciones

RUTA_EXCEL = BASE_DIR / "data" / "templates" / "prueba.xlsx"
X, y = intenciones(RUTA_EXCEL)

modelos = {
    "SVM lineal": make_pipeline(TfidfVectorizer(), LinearSVC()),
    "Naive Bayes multinomial": make_pipeline(
        TfidfVectorizer(), MultinomialNB()
    ),
}

particiones = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

for nombre, modelo in modelos.items():
    metricas = cross_validate(
        modelo,
        X,
        y,
        cv=particiones,
        scoring=["accuracy", "f1_macro"],
    )

    resultados.append({
        "Modelo": nombre,
        "Exactitud promedio": metricas["test_accuracy"].mean(),
        "F1 macro promedio": metricas["test_f1_macro"].mean(),
    })

pd.DataFrame(resultados).round(3)

,Modelo,Exactitud promedio,F1 macro promedio
0,SVM lineal,0.85,0.849
1,Naive Bayes multinomial,0.80,0.798


In [17]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

etiquetas = sorted(y.unique())

for nombre, modelo in modelos.items():
    predicciones = cross_val_predict(
        modelo,
        X,
        y,
        cv=particiones,
    )

    matriz = pd.DataFrame(
        confusion_matrix(y, predicciones, labels=etiquetas),
        index=pd.Index(etiquetas, name="Intención real"),
        columns=pd.Index(etiquetas, name="Intención predicha"),
    )

    evaluacion = pd.DataFrame({
        "Pregunta": X.to_numpy(),
        "Intención real": y.to_numpy(),
        "Intención predicha": predicciones,
    })

    errores = evaluacion[
        evaluacion["Intención real"] != evaluacion["Intención predicha"]
    ]

    print(nombre)
    display(matriz)
    display(errores)

SVM lineal


Intención predicha,consultar_alimentacion,consultar_tamano
Intención real,,
consultar_alimentacion,17,3
consultar_tamano,3,17


,Pregunta,Intención real,Intención predicha
4,¿Era herbívoro o carnívoro?,consultar_alimentacion,consultar_tamano
5,Me gustaría conocer sus dimensiones.,consultar_tamano,consultar_alimentacion
14,Me interesa saber si comía carne.,consultar_alimentacion,consultar_tamano
20,Cuéntame cómo era su alimentación.,consultar_alimentacion,consultar_tamano
25,¿Qué distancia había desde el hocico hasta la ...,consultar_tamano,consultar_alimentacion
27,¿De qué longitud era su cuerpo?,consultar_tamano,consultar_alimentacion


Naive Bayes multinomial


Intención predicha,consultar_alimentacion,consultar_tamano
Intención real,,
consultar_alimentacion,16,4
consultar_tamano,4,16


,Pregunta,Intención real,Intención predicha
4,¿Era herbívoro o carnívoro?,consultar_alimentacion,consultar_tamano
5,Me gustaría conocer sus dimensiones.,consultar_tamano,consultar_alimentacion
6,Quiero conocer su dieta.,consultar_alimentacion,consultar_tamano
14,Me interesa saber si comía carne.,consultar_alimentacion,consultar_tamano
20,Cuéntame cómo era su alimentación.,consultar_alimentacion,consultar_tamano
25,¿Qué distancia había desde el hocico hasta la ...,consultar_tamano,consultar_alimentacion
27,¿De qué longitud era su cuerpo?,consultar_tamano,consultar_alimentacion
39,¿Qué medidas se estiman para un adulto?,consultar_tamano,consultar_alimentacion
